# PBjam and Mimir comparison for one star

This notebook compares the legacy `pbjam.IO.psd` calculation with the corresponding Mimir workflow for **KIC 8006161**. The comparison is deliberately controlled:

1. Mimir downloads and reduces one Kepler short-cadence quarter;
2. the same time, flux, and uncertainty arrays are passed to both packages;
3. both spectra use the same oversampling factor; and
4. the resulting frequency grids and power-density spectra are compared.

Using one shared light curve matters. If each package downloads and cleans the data independently, preprocessing differences can be mistaken for differences in the periodogram.

From the repository root, install the comparison dependencies with:

```bash
python -m pip install -e ".[mast,compat]" matplotlib
```


In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import pbjam
import pbjam.IO as pbjam_io

from mimir import load_lightcurve, power_spectrum

plt.rcParams["figure.figsize"] = (9, 4)

print(f"PBjam version: {getattr(pbjam, '__version__', 'unknown')}")


## Select the target and data product

A single quarter keeps the example reasonably quick and avoids hiding the comparison behind a multi-quarter download. `numax` is supplied only to choose Mimir's PBjam-compatible flattening window; it is not used by either power-spectrum calculation.

If Quarter 10 is unavailable in a future archive query, choose any returned short-cadence quarter from `mimir.search_lightcurves(TARGET, **SEARCH_KWARGS)`.


In [ ]:
TARGET = "KIC 8006161"
NUMAX_UHZ = 3500.0
OVERSAMPLING = 1

SEARCH_KWARGS = {
    "mission": "Kepler",
    "author": "Kepler",
    "exptime": 60,
    "quarter": 10,
}


## Download and reduce the light curve once

`load_lightcurve` performs the Lightkurve-backed search, download, stitch, normalization, outlier rejection, flattening, and conversion to relative flux in ppm. The returned `TimeSeries` is the common input for both packages.


In [ ]:
series = load_lightcurve(
    TARGET,
    search_kwargs=SEARCH_KWARGS,
    numax=NUMAX_UHZ,
    outlier_sigma=5.0,
    flatten=True,
    ppm=True,
)

{
    "samples": series.n_samples,
    "duration (d)": series.duration,
    "cadence (s)": series.cadence * 86400.0,
    "duty cycle": series.duty_cycle,
    "flux unit": series.flux_unit,
}


In [ ]:
fig, ax = plt.subplots()
ax.plot(series.time, series.flux, ".", ms=0.7, alpha=0.6)
ax.set(
    xlabel=f"Time [{series.time_unit}]",
    ylabel=f"Relative flux [{series.flux_unit}]",
    title=f"{TARGET}: shared reduced light curve",
)
plt.show()


## Calculate the PBjam spectrum

This is the legacy pattern used by PBjam: construct `pbjam.IO.psd`, then call the object to calculate the periodogram. Passing arrays prevents PBjam from making a second MAST request or applying a different reduction.

`method="fast"` selects PBjam's fast Astropy Lomb–Scargle route. Mimir instead calls `nifty-ls` directly.


In [ ]:
pbjam_spectrum = pbjam_io.psd(
    TARGET,
    time=series.time,
    flux=series.flux,
    flux_err=series.flux_err,
)
pbjam_spectrum(oversampling=OVERSAMPLING, method="fast")

pbjam_frequency = np.asarray(pbjam_spectrum.freq, dtype=float)
pbjam_power_density = np.asarray(pbjam_spectrum.powerdensity, dtype=float)


## Calculate the Mimir spectrum

Mimir accepts its validated `TimeSeries` directly. It returns an immutable result object rather than mutating a callable spectrum object.


In [ ]:
mimir_spectrum = power_spectrum(
    time_series=series,
    oversampling=OVERSAMPLING,
    nyquist_factor=1.0,
    frequency_unit="uHz",
)

mimir_frequency = mimir_spectrum.frequency
mimir_power_density = mimir_spectrum.power_density

print(f"Mimir nifty-ls backend: {mimir_spectrum.backend}")


## Compare the numerical summaries

Power density is the cleanest quantity to compare because its integral has the same physical interpretation in both packages. With frequency in μHz and flux in ppm, its unit is ppm²/μHz.

The sums below use each returned grid's measured bin spacing. For a complete one-sided spectrum, the integrated power density should be close to the variance of the shared light curve.


In [ ]:
def spectrum_summary(frequency, power_density, *, numax, half_width=1000.0):
    """Summarize one power-density spectrum on its native grid."""
    spacing = float(np.median(np.diff(frequency)))
    band = np.abs(frequency - numax) <= half_width
    peak_index = np.flatnonzero(band)[np.argmax(power_density[band])]
    return {
        "bins": frequency.size,
        "spacing (uHz)": spacing,
        "maximum frequency (uHz)": float(frequency[-1]),
        "integrated PSD (ppm^2)": float(np.sum(power_density) * spacing),
        "peak near numax (uHz)": float(frequency[peak_index]),
        "power around numax (ppm^2)": float(
            np.sum(power_density[band]) * spacing
        ),
    }


summaries = {
    "light-curve variance (ppm^2)": float(np.var(series.flux)),
    "PBjam": spectrum_summary(
        pbjam_frequency,
        pbjam_power_density,
        numax=NUMAX_UHZ,
    ),
    "Mimir": spectrum_summary(
        mimir_frequency,
        mimir_power_density,
        numax=NUMAX_UHZ,
    ),
}
summaries


## Overlay the spectra

The upper panel shows the oscillation region. The lower panel interpolates PBjam onto Mimir's grid and plots the ratio after modest smoothing. Smoothing suppresses the point-to-point exponential scatter and makes broad normalization or high-frequency trends easier to see; it is used only for visualization.


In [ ]:
def boxcar(values, width=101):
    """Return a boxcar-smoothed copy of a one-dimensional array."""
    kernel = np.ones(width) / width
    return np.convolve(values, kernel, mode="same")


pbjam_on_mimir_grid = np.interp(
    mimir_frequency,
    pbjam_frequency,
    pbjam_power_density,
    left=np.nan,
    right=np.nan,
)
smooth_pbjam = boxcar(pbjam_on_mimir_grid)
smooth_mimir = boxcar(mimir_power_density)
ratio = smooth_mimir / smooth_pbjam

fig, (ax, ratio_ax) = plt.subplots(
    2,
    1,
    figsize=(10, 7),
    sharex=True,
    gridspec_kw={"height_ratios": [3, 1]},
)
ax.plot(pbjam_frequency, pbjam_power_density, lw=0.5, alpha=0.45, label="PBjam")
ax.plot(mimir_frequency, mimir_power_density, lw=0.5, alpha=0.45, label="Mimir")
ax.plot(mimir_frequency, smooth_pbjam, lw=1.5, label="PBjam, smoothed")
ax.plot(mimir_frequency, smooth_mimir, lw=1.5, label="Mimir, smoothed")
ax.set(
    ylabel="Power density [ppm²/μHz]",
    title=f"{TARGET}: PBjam and Mimir power-density spectra",
    yscale="log",
)
ax.legend(ncol=2)

ratio_ax.axhline(1.0, color="0.3", ls="--")
ratio_ax.plot(mimir_frequency, ratio, color="tab:purple", lw=1.0)
ratio_ax.set(
    xlabel="Frequency [μHz]",
    ylabel="Mimir / PBjam",
    xlim=(1500.0, 5000.0),
    ylim=(0.5, 1.5),
)
plt.tight_layout()
plt.show()


## Compare the high-frequency behaviour

One reason Mimir uses `nifty-ls` is to avoid the high-frequency bias seen in the Astropy Lomb–Scargle implementation used by PBjam. The following plot shows the smoothed ratio over the entire shared frequency range. A frequency-dependent departure from one is more informative here than small differences in individual noisy bins.


In [ ]:
shared = np.isfinite(ratio) & (smooth_pbjam > 0.0)

fig, ax = plt.subplots(figsize=(10, 4))
ax.axhline(1.0, color="0.3", ls="--")
ax.plot(mimir_frequency[shared], ratio[shared], color="tab:purple", lw=0.8)
ax.set(
    xlabel="Frequency [μHz]",
    ylabel="Smoothed Mimir / PBjam",
    title="Frequency dependence of the PSD ratio",
    ylim=(0.5, 1.5),
)
plt.show()


## Interpreting the comparison

The two spectra should recover the same oscillation envelope and broadly similar integrated power because they use the same samples. They are not expected to be identical bin by bin:

- PBjam uses Astropy's Lomb–Scargle implementation; Mimir uses `nifty-ls`.
- Mimir explicitly normalizes the physical one-sided band through Nyquist.
- Mimir uses inverse-variance-weighted centring when uncertainties are supplied, whereas PBjam 2.0.4 centres on the unweighted mean.
- PBjam's legacy `amplitude` attribute has a different, dimensionally inconsistent definition, so this notebook compares power density instead.
- Small endpoint and bin-count differences can occur even when the nominal Fourier spacing agrees.

For migration testing, the important checks are therefore agreement in the recovered astrophysical features, integrated power, and broad spectral shape—not exact equality of every periodogram bin.
